# HydroSense-Kenya — Level 1
**ICS 2207 Scientific Computing**

Level 1: Scientific Problem Framing and Python Foundation

## Problem Statement

Agriculture uses a lot of water, and in Kenya this is a serious issue especially for smallholder farmers who rely on seasonal rainfall and irrigation to grow their crops. Many farms in peri-urban areas around Nairobi and in semi-arid counties like Machakos and Kajiado face the challenge of managing water carefully — too little and crops die, too much and you waste water and energy running pumps unnecessarily.

The problem we are trying to solve in this project is: given daily weather data and soil sensor readings from a demonstration farm, how do we model the water balance of the soil and use that to make better irrigation decisions?

The central scientific question from the project brief is:
> *Given weather and soil-sensor data, how can we model water availability, estimate water deficit, simulate future soil moisture, and recommend an efficient irrigation plan that minimizes water use without exposing crops to moisture stress?*

To answer this we need to track how soil moisture changes each day. The model we are building is based on a discrete water balance equation where the soil moisture at the next time step depends on the current moisture, how much it rained, how much we irrigated, how much water was lost to evapotranspiration, and how much drained away:

`S(t+1) = S(t) + R(t) + I(t) - ET(t) - D(t)`

We also need to estimate evapotranspiration (ET) which is the water lost from the soil through evaporation and plant transpiration. For this project we use a simplified formula:

`ET = max(0, 0.12*T + 0.35*W + 2.4*Solar - 0.025*H)`

where T is temperature, W is wind speed, Solar is the solar radiation index and H is humidity.

The farm has three crop zones — Zone A grows tomatoes, Zone B grows kale, and Zone C grows maize. Each zone has different moisture requirements and drainage characteristics. We have three datasets to work with: daily weather records, daily soil sensor readings for each zone, and crop zone parameters that tell us the moisture thresholds and field capacity for each zone.

The goal of this level is to load these datasets, understand their structure, write the core functions we will need throughout the project, and produce a first basic visualization.

### Assumptions
- The ET formula is a simplified empirical estimate, not a full agronomic model like Penman-Monteith.
- We assume rainfall fully infiltrates the soil with no surface runoff.
- Each zone is treated as spatially uniform — one sensor reading represents the whole zone.
- The drainage coefficient is constant and does not vary with soil wetness.
- Sensor readings taken at noon are representative of the full day.

### Limitations
- The dataset only covers one month (March 2026) so we cannot capture seasonal patterns.
- Crop growth stages are not modelled — water demand in reality changes as crops mature.
- There is no groundwater term in the water balance model.


## Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

print('libraries loaded')

libraries loaded


## Loading the Datasets

We load all three CSV files using pandas. The `na_values` argument makes sure that both empty cells and cells with the string 'NA' are treated as missing values.

In [11]:
weather = pd.read_csv('/kaggle/input/datasets/ashleybrightone/hydrosense-kenya/weather_daily.csv', na_values=['NA', ''])
soil    = pd.read_csv('/kaggle/input/datasets/ashleybrightone/hydrosense-kenya/soil_sensor_data.csv', na_values=['NA', ''])
params  = pd.read_csv('/kaggle/input/datasets/ashleybrightone/hydrosense-kenya/crop_zone_parameters.csv', na_values=['NA', ''])

weather['date'] = pd.to_datetime(weather['date'])

print('weather shape:', weather.shape)
print('soil shape:', soil.shape)
print('params shape:', params.shape)

weather shape: (30, 6)
soil shape: (90, 7)
params shape: (3, 7)


### Quick look at each dataset

In [12]:
print('--- weather_daily ---')
display(weather.head())

print('--- soil_sensor_data ---')
display(soil.head(6))

print('--- crop_zone_parameters ---')
display(params)

--- weather_daily ---


,date,rainfall_mm,temperature_c,humidity_pct,wind_speed_mps,solar_index
0,2026-03-01,3.2,23.8,69.7,2.28,0.78
1,2026-03-02,2.2,25.9,62.0,1.62,0.71
2,2026-03-03,3.0,26.8,64.4,2.11,0.75
3,2026-03-04,1.6,27.0,64.6,2.09,0.58
4,2026-03-05,23.7,26.6,61.0,1.36,0.62


--- soil_sensor_data ---


,timestamp,zone_id,soil_moisture_pct,tank_level_liters,pump_flow_lpm,pump_power_watts,sensor_status
0,2026-03-01 12:00,Zone_A,33.2,4829,18.8,437,OK
1,2026-03-01 12:00,Zone_B,34.7,4728,22.1,461,OK
2,2026-03-01 12:00,Zone_C,28.2,4515,25.7,490,OK
3,2026-03-02 12:00,Zone_A,36.1,4757,16.6,411,OK
4,2026-03-02 12:00,Zone_B,32.0,4645,20.4,445,OK
5,2026-03-02 12:00,Zone_C,31.0,4519,24.6,492,OK


--- crop_zone_parameters ---


,zone_id,crop_type,area_m2,min_moisture_pct,target_moisture_pct,field_capacity_pct,drainage_coefficient
0,Zone_A,tomato,120,22,33,41,0.18
1,Zone_B,kale,90,24,35,43,0.15
2,Zone_C,maize,180,20,31,40,0.22


### Checking for missing values

The brief says the datasets contain deliberate missing values and anomalies for us to handle. Let's confirm what we are dealing with before doing any cleaning.

In [13]:
print('Missing values in weather_daily:')
print(weather.isnull().sum())
print()
print('Missing values in soil_sensor_data:')
print(soil.isnull().sum())

Missing values in weather_daily:
date              0
rainfall_mm       1
temperature_c     0
humidity_pct      1
wind_speed_mps    0
solar_index       0
dtype: int64

Missing values in soil_sensor_data:
timestamp            0
zone_id              0
soil_moisture_pct    1
tank_level_liters    0
pump_flow_lpm        0
pump_power_watts     0
sensor_status        0
dtype: int64


## Data Dictionary

Below is a description of every column across all three datasets.

### weather_daily.csv

| Column | Units | Type | Description |
|--------|-------|------|-------------|
| date | — | datetime | Calendar date of the observation |
| rainfall_mm | mm | float | Total daily rainfall |
| temperature_c | °C | float | Mean daily air temperature |
| humidity_pct | % | float | Mean daily relative humidity |
| wind_speed_mps | m/s | float | Mean daily wind speed |
| solar_index | 0–1 | float | Normalised solar radiation index |

**Data quality notes:** rainfall_mm has 1 missing value (Mar 8), humidity_pct has 1 missing value (Mar 21), temperature_c has an outlier of 45.8°C on Mar 14 which is not physically realistic for Nairobi in March, and rainfall on Mar 26 is 85mm which is unusually high and flagged for review.

### soil_sensor_data.csv

| Column | Units | Type | Description |
|--------|-------|------|-------------|
| timestamp | — | object | Date and time of sensor reading |
| zone_id | — | object | Farm zone (Zone_A, Zone_B, Zone_C) |
| soil_moisture_pct | % | float | Volumetric soil moisture at root zone |
| tank_level_liters | L | int | Water remaining in the storage tank |
| pump_flow_lpm | L/min | float | Flow rate from the irrigation pump |
| pump_power_watts | W | int | Power consumed by the pump |
| sensor_status | — | object | Sensor health flag: OK or CHECK |

**Data quality notes:** Zone_B has a missing soil_moisture_pct on Mar 6, Zone_C has an impossible tank level of 9900L on Mar 14, Zone_B has a pump_flow_lpm of 0.0 with a CHECK status on Mar 21, and Zone_B has a soil_moisture_pct of 8.5% on Mar 25 which is below wilting point and likely a sensor fault.

### crop_zone_parameters.csv

| Column | Units | Type | Description |
|--------|-------|------|-------------|
| zone_id | — | object | Zone identifier |
| crop_type | — | object | Crop grown in the zone |
| area_m2 | m² | int | Zone area |
| min_moisture_pct | % | int | Minimum soil moisture before crop stress |
| target_moisture_pct | % | int | Target moisture after irrigation |
| field_capacity_pct | % | int | Maximum moisture before drainage occurs |
| drainage_coefficient | — | float | Fraction of excess moisture lost per day |


## Core Functions

These two functions are the foundation of the whole model. We write them in plain Python here without NumPy — the vectorised versions come in Level 2.

In [15]:
def evapotranspiration(T, W, Solar, H):
    """
    Estimate daily evapotranspiration in mm/day.
    Uses the simplified formula from the project brief (section 2.3).
    
    Parameters:
        T     : temperature in degrees C
        W     : wind speed in m/s
        Solar : solar index (0 to 1)
        H     : relative humidity in %
    Returns:
        ET in mm/day, minimum 0
    """
    return max(0.0, 0.12*T + 0.35*W + 2.4*Solar - 0.025*H)


def water_balance(S_t, R_t, I_t, ET_t, D_t):
    """
    Compute next-day soil moisture using the discrete water balance equation.
    S(t+1) = S(t) + R(t) + I(t) - ET(t) - D(t)
    
    Parameters:
        S_t  : current soil moisture (%)
        R_t  : rainfall (mm)
        I_t  : irrigation applied (mm)
        ET_t : evapotranspiration loss (mm/day)
        D_t  : drainage loss
    Returns:
        soil moisture at t+1
    """
    return S_t + R_t + I_t - ET_t - D_t


def compute_drainage(S_t, field_capacity, drainage_coeff):
    """
    Drainage only happens when soil moisture exceeds field capacity.
    D = drainage_coeff * max(0, S_t - field_capacity)
    """
    return drainage_coeff * max(0.0, S_t - field_capacity)


print('functions defined')

functions defined


### Testing the functions

Before using these functions on real data we want to verify they give sensible results on a few manual examples.

In [17]:
# ET on a hot dry windy day should be high
et1 = evapotranspiration(T=30.0, W=3.5, Solar=0.90, H=40.0)
print('ET hot day:', round(et1, 3), 'mm/day')

# ET on a cool humid overcast day should be low
et2 = evapotranspiration(T=18.0, W=0.5, Solar=0.20, H=90.0)
print('ET cool day:', round(et2, 3), 'mm/day')

# ET should never go negative
et3 = evapotranspiration(T=5.0, W=0.1, Solar=0.05, H=99.0)
print('ET floor test:', et3, '(should be 0.0)')

# Water balance: soil gains moisture on a rainy day with no irrigation
s_next = water_balance(S_t=28.0, R_t=10.0, I_t=0.0, ET_t=3.5, D_t=0.0)
print('Water balance (gain):', s_next, '% (expected 34.5)')

# Drainage test: moisture above field capacity should produce drainage
d = compute_drainage(S_t=45.0, field_capacity=41.0, drainage_coeff=0.18)
print('Drainage above capacity:', round(d, 4), '(expected 0.72)')

ET hot day: 5.985 mm/day
ET cool day: 0.565 mm/day
ET floor test: 0.0 (should be 0.0)
Water balance (gain): 34.5 % (expected 34.5)
Drainage above capacity: 0.72 (expected 0.72)


### Computing ET for the full weather dataset

We loop through every row and compute daily ET. Missing values are temporarily forward-filled just for this step — proper cleaning happens in Level 4.

In [21]:
weather_temp = weather.copy()
weather_temp[['temperature_c','wind_speed_mps','solar_index','humidity_pct']] = (
    weather_temp[['temperature_c','wind_speed_mps','solar_index','humidity_pct']]
    .ffill().bfill()
)

et_values = []
for _, row in weather_temp.iterrows():
    et = evapotranspiration(
        T=row['temperature_c'],
        W=row['wind_speed_mps'],
        Solar=row['solar_index'],
        H=row['humidity_pct']
    )
    et_values.append(et)

weather_temp['ET_mm'] = et_values

print('ET summary:')
print('  mean:', round(weather_temp['ET_mm'].mean(), 3), 'mm/day')
print('  min: ', round(weather_temp['ET_mm'].min(), 3), 'mm/day')
print('  max: ', round(weather_temp['ET_mm'].max(), 3), 'mm/day')

ET summary:
  mean: 3.749 mm/day
  min:  2.647 mm/day
  max:  5.982 mm/day


## Visualisation

### Rainfall — March 2026

A basic bar chart of daily rainfall. We can already see the missing value on Mar 8, the extreme event on Mar 26, and that most of the month had low rainfall.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.bar(weather['date'], weather['rainfall_mm'].fillna(0),
       color='steelblue', edgecolor='white', linewidth=0.5)

mean_rain = weather['rainfall_mm'].mean()
ax.axhline(mean_rain, color='black', linestyle='--', linewidth=1,
           label='mean: ' + str(round(mean_rain, 1)) + ' mm')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
plt.setp(ax.get_xticklabels(), rotation=40, ha='right', fontsize=9)

ax.set_title('Daily Rainfall - Nairobi Demo Farm, March 2026')
ax.set_xlabel('Date')
ax.set_ylabel('Rainfall (mm)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/rainfall_plot.png', dpi=150)
plt.show()

### Rainfall vs ET

Comparing rainfall and ET helps us see which days had a water surplus and which had a deficit. Days where ET is higher than rainfall are days the crops needed irrigation.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.bar(weather_temp['date'], weather_temp['rainfall_mm'].fillna(0),
        color='steelblue', label='Rainfall (mm)')
ax1.set_ylabel('Rainfall (mm)')
ax1.set_title('Rainfall vs Evapotranspiration - March 2026')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2.plot(weather_temp['date'], weather_temp['ET_mm'],
         color='tomato', linewidth=1.8, marker='o', markersize=3, label='ET (mm/day)')
ax2.set_ylabel('ET (mm/day)')
ax2.set_xlabel('Date')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax2.xaxis.set_major_locator(mdates.DayLocator(interval=3))
plt.setp(ax2.get_xticklabels(), rotation=40, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/rainfall_vs_et_plot.png', dpi=150)
plt.show()

## Descriptive Statistics

In [ ]:
print('Weather dataset summary:')
display(weather.describe().round(2))

print('Zone parameters:')
display(params)

## Assumptions and Limitations

### Assumptions
1. The ET formula is empirical and simplified — it is not meant to replace a proper agronomic model.
2. All rainfall is assumed to enter the soil. In reality some would run off, especially during heavy events like the 85mm day.
3. Each zone is treated as a single uniform unit. Soil moisture in reality varies across the zone.
4. The drainage coefficient is constant throughout the month.
5. A noon sensor reading is taken to represent the soil moisture for the full day.

### Limitations
1. One month of data is not enough to draw conclusions about seasonal patterns or long-term behaviour.
2. The model does not account for how crop water demand changes as the plant grows.
3. There is no groundwater or capillary rise term in the water balance.
4. The ET formula has not been validated against measured values for this specific location.
